# B2-024-gpu-scientific-ml-capstone — Practice p08 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** open-ended-model-evaluation

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch

y_true = torch.tensor([0, 1, 2, 1, 0, 2, 2, 1, 0, 1, 2, 0])
pred_a = torch.tensor([0, 1, 2, 2, 0, 2, 1, 1, 0, 1, 2, 1])
pred_b = torch.tensor([0, 2, 2, 2, 1, 2, 1, 1, 0, 0, 2, 1])
N_BOOT = 2000
ALPHA = 0.1
ABS_TOL = 1e-9

## Solution

The percentile bootstrap treats the $n$ examples as the population: it draws $n$ example indices with replacement, recomputes the metric on that resample, repeats $B$ times, and reads the $\alpha/2$ and $1-\alpha/2$ quantiles of the replicate statistics.
Two details are pinned by the contract:

* the whole index matrix `idx` of shape `(n_boot, n)` is drawn **once** from the passed generator, so the result is a deterministic function of the generator's seed;
* labels and predictions are resampled **together** (`y_true[idx[b]]`, `y_pred[idx[b]]`), because the unit being resampled is an *example* (a label–prediction pair).

For the paired difference, the **same** `idx` row is applied to both models, so each replicate compares the two models on the same resampled examples.

In [ ]:
def _quantiles(stats, alpha):
    stats = torch.tensor(stats, dtype=torch.float64)
    q = torch.tensor([alpha / 2, 1 - alpha / 2], dtype=torch.float64)
    low, high = torch.quantile(stats, q).tolist()
    return float(low), float(high)


def bootstrap_ci(metric_fn, y_true, y_pred, n_boot, alpha, generator):
    point = float(metric_fn(y_true, y_pred))
    n = y_true.shape[0]
    idx = torch.randint(0, n, (n_boot, n), generator=generator)
    stats = [float(metric_fn(y_true[idx[b]], y_pred[idx[b]])) for b in range(n_boot)]
    low, high = _quantiles(stats, alpha)
    return point, low, high


def paired_bootstrap_diff(metric_fn, y_true, pred_a, pred_b, n_boot, alpha, generator):
    point = float(metric_fn(y_true, pred_a)) - float(metric_fn(y_true, pred_b))
    n = y_true.shape[0]
    idx = torch.randint(0, n, (n_boot, n), generator=generator)
    stats = [float(metric_fn(y_true[idx[b]], pred_a[idx[b]])) - float(metric_fn(y_true[idx[b]], pred_b[idx[b]]))
             for b in range(n_boot)]
    low, high = _quantiles(stats, alpha)
    return point, low, high


def wrong_bootstrap_ci(metric_fn, y_true, y_pred, n_boot, alpha, generator):
    # Wrong variant: resamples only the predictions, breaking the label-prediction pairing.
    point = float(metric_fn(y_true, y_pred))
    n = y_true.shape[0]
    idx = torch.randint(0, n, (n_boot, n), generator=generator)
    stats = [float(metric_fn(y_true, y_pred[idx[b]])) for b in range(n_boot)]
    low, high = _quantiles(stats, alpha)
    return point, low, high


accuracy = lambda t, p: (t == p).double().mean()
gen = lambda: torch.Generator().manual_seed(7)

ci_a = bootstrap_ci(accuracy, y_true, pred_a, N_BOOT, ALPHA, gen())
ci_b = bootstrap_ci(accuracy, y_true, pred_b, N_BOOT, ALPHA, gen())
diff = paired_bootstrap_diff(accuracy, y_true, pred_a, pred_b, N_BOOT, ALPHA, gen())
ci_a_again = bootstrap_ci(accuracy, y_true, pred_a, N_BOOT, ALPHA, gen())
diff_again = paired_bootstrap_diff(accuracy, y_true, pred_a, pred_b, N_BOOT, ALPHA, gen())
wrong = wrong_bootstrap_ci(accuracy, y_true, pred_a, N_BOOT, ALPHA, gen())
print("A:", ci_a)
print("B:", ci_b)
print("A - B paired:", diff)
print("wrong (predictions only):", wrong)

**Probe 6 — what the wrong variant measures.**
Resampling only the predictions pairs each true label with a *random* prediction from the list, so a replicate's accuracy measures how often a randomly drawn prediction happens to equal a fixed label: it is (approximately) the agreement expected by chance from the two label distributions, not the model's accuracy.
Its interval is centred near that chance level (here about $0.33$, far below the observed $0.75$) and says nothing about how uncertain the model's accuracy is.

**Probe 7 — why the paired interval excludes 0 though the separate intervals overlap.**
The separate intervals each include the variation from *which* examples happen to be in the resample, and most of that variation is shared: an easy resample raises both models' accuracies, a hard one lowers both.
The paired bootstrap applies the same resample to both models, so this shared variation cancels in the difference and only the examples on which the two models *disagree* move the replicate difference.
Here A is right on every example on which B is right and on three more (indices 1, 4, 9), and B is never right where A is wrong, so every replicate difference is $\ge 0$ and it is positive whenever a resample contains one of those three examples; the two marginal intervals, by contrast, are wide enough to overlap. (With label frequencies $(4,4,4)/12$ and prediction frequencies $(3,5,4)/12$, the chance agreement is $(12+20+16)/144=1/3$, the centre of the wrong variant's interval.)

In [ ]:
disagree_a_only = ((pred_a == y_true) & (pred_b != y_true)).nonzero().flatten().tolist()
disagree_b_only = ((pred_b == y_true) & (pred_a != y_true)).nonzero().flatten().tolist()
print("A right, B wrong:", disagree_a_only, "| B right, A wrong:", disagree_b_only)

### Answer check

In [ ]:
def close(u, v):
    return all(math.isclose(a, b, rel_tol=0, abs_tol=ABS_TOL) for a, b in zip(u, v))


# Probes 1-3
assert close(ci_a, (0.75, 0.5, 0.9166666666666666))
assert close(ci_b, (0.5, 0.25, 0.75))
assert close(diff, (0.25, 0.08333333333333326, 0.49999999999999994))
# Probe 4
assert all(type(v) is float for v in ci_a + ci_b + diff)
assert ci_a[1] <= ci_a[0] <= ci_a[2] and ci_b[1] <= ci_b[0] <= ci_b[2]
# Probe 5
assert ci_a == ci_a_again and diff == diff_again
# Probe 6
assert close(wrong[1:], (0.08333333333333333, 0.5833333333333334))
assert (wrong[1], wrong[2]) != (ci_a[1], ci_a[2])
# Probe 7 support: the paired interval excludes 0 while the marginal intervals overlap
assert diff[1] > 0 and ci_b[2] >= ci_a[1]
assert disagree_a_only == [1, 4, 9] and disagree_b_only == []